In [10]:
import numpy as np
import pandas as pd
import re
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import requests

from tqdm import tqdm

In [11]:
start=datetime(2021,1,1,0)
end=datetime(2022,10,13,0)
# end=datetime.now()
times=[start];

while times[-1]<end:
    times.append(times[-1]+timedelta(hours=4)); 

In [12]:
ARCHIVER_URL = 'http://lcls-archapp.slac.stanford.edu/retrieval/data/getData.json'
TIMEOUT_SECONDS = 20.0

DEFAULT_UND_CELLS = [26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]
KAct = [f'USEG:UNDS:{cell:02d}50:KAct' for cell in DEFAULT_UND_CELLS]
PVs=KAct
# PVs

In [13]:
import ipykernel
ipykernel.__version__

'6.29.5'

In [14]:
import sys
sys.version

'3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:45:31) [GCC 13.3.0]'

In [15]:
# pull undulator K values from the archive for each time in times
# first restore any cached rows from kvals.csv so we only fetch missing times
# use one threadpool whose workers draw from missing times; advance the progress bar as jobs finish
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from tqdm.auto import tqdm

def _to_utc_str(t):
    return t.strftime('%Y-%m-%dT%H:%M:%S.000Z')

def _fetch_archiver_snapshot(pv_names, from_time, to_time):
    params = [('pv', pv) for pv in pv_names]
    params.extend([('from', _to_utc_str(from_time)), ('to', _to_utc_str(to_time))])
    response = requests.get(ARCHIVER_URL, params=params, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    payload = response.json()
    latest = {}
    for entry in payload:
        pv_name = entry.get('meta', {}).get('name')
        data = entry.get('data', [])
        if not pv_name or not data:
            continue
        latest[pv_name] = data[-1].get('val', np.nan)
    return latest

def fetch_kvals(ii, ft):
    latest = _fetch_archiver_snapshot(PVs, ft, ft + timedelta(seconds=3))
    Ks = np.array([latest.get(pv, np.nan) for pv in PVs], dtype=float)
    return ii, ft, Ks

cache_path = Path('kvals.csv')
cached_kvals_by_time = {}
cached_und_cells = None

if cache_path.exists():
    lines = [line.strip() for line in cache_path.read_text().splitlines() if line.strip()]
    if lines:
        header = lines[0].split(',')
        if len(header) > 1:
            cached_und_cells = np.array([float(value) for value in header[1:]])
        for line in lines[1:]:
            parts = line.split(',')
            if len(parts) < 2:
                continue
            try:
                dt = datetime.strptime(parts[0], '%Y-%m-%d %H:%M:%S')
                kval = np.array([float(value) for value in parts[1:]])
            except ValueError:
                continue
            if cached_und_cells is not None and len(kval) != len(cached_und_cells):
                continue
            cached_kvals_by_time[dt] = kval

requested_times = list(times)
results = [None] * len(requested_times)
archive_data = None
missing = []

for ii, ft in enumerate(requested_times):
    cached_kvals = cached_kvals_by_time.get(ft)
    if cached_kvals is not None:
        results[ii] = (ft, cached_kvals)
    else:
        missing.append((ii, ft))

if missing:
    with ThreadPoolExecutor(max_workers=min(8, len(missing) or 1)) as executor:
        futures = [executor.submit(fetch_kvals, ii, ft) for ii, ft in missing]
        for future in tqdm(as_completed(futures), total=len(futures), desc='Fetching K values'):
            try:
                ii, ft, Ks = future.result()
                results[ii] = (ft, Ks)
                cached_kvals_by_time[ft] = Ks
            except Exception:
                pass
                # print(f'Error fetching data for time {ft}: {exc}')

pairs = [result for result in results if result is not None]
times = [ft for ft, _ in pairs]
kvals = [kv for _, kv in pairs]
all_kvals_by_time = dict(sorted(cached_kvals_by_time.items()))

In [16]:
if PVs:
    und_cells=np.array([float(re.search(':([0-9][0-9])[0-9][0-9]:', pv)[1]) for pv in PVs])
elif cached_und_cells is not None:
    und_cells=cached_und_cells
else:
    raise RuntimeError('Could not determine undulator cells from archive data or kvals.csv cache')

In [17]:
# combine cached and newly fetched rows, then build a DataFrame for display

all_kvals_by_time.update({dt: k for dt, k in zip(times, kvals)})

sorted_times = sorted(all_kvals_by_time)
kvals_df = pd.DataFrame(
    data=[all_kvals_by_time[dt] for dt in sorted_times],
    index=sorted_times,
    columns=[str(cell) for cell in und_cells],
)
kvals_df.index.name = 'Datetime'
kvals_df

,26.0,27.0,28.0,29.0,30.0,31.0,32.0,33.0,34.0,36.0,...,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0
Datetime,,,,,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00,5.072503,5.072384,5.072592,5.072143,5.072183,5.072640,5.071885,5.072186,5.072337,5.071045,...,5.068892,5.065380,5.060853,5.055229,5.047342,5.038976,5.028820,5.017229,5.004429,4.991130
2021-01-01 04:00:00,5.072414,5.072325,5.072533,5.072114,5.072125,5.072581,5.071826,5.072127,5.072278,5.070986,...,5.068863,5.065321,5.060795,5.055174,5.047313,5.038946,5.028762,5.017170,5.004400,4.991072
2021-01-01 08:00:00,5.072419,5.072266,5.072474,5.072054,5.072095,5.072522,5.071796,5.072068,5.072219,5.070927,...,5.068834,5.065292,5.060765,5.055106,5.047225,5.038888,5.028732,5.017112,5.004342,4.991043
2021-01-01 12:00:00,5.072537,5.072325,5.072533,5.072114,5.072125,5.072581,5.071826,5.072098,5.072249,5.070957,...,5.068834,5.065321,5.060795,5.055165,5.047283,5.038917,5.028762,5.017141,5.004371,4.991043
2021-01-01 16:00:00,5.072597,5.072384,5.072592,5.072143,5.072154,5.072640,5.071855,5.072127,5.072308,5.071016,...,5.068892,5.065380,5.060824,5.055227,5.047313,5.038946,5.028791,5.017229,5.004429,4.991130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-10-12 08:00:00,3.637687,3.637440,3.636439,3.651557,3.664285,3.678871,3.692643,3.706774,3.720040,3.690988,...,3.696030,3.696983,3.682046,3.695314,3.715444,2.714202,2.349474,2.599818,3.177089,2.814965
2022-10-12 12:00:00,3.638056,3.637998,3.636802,3.651747,3.664643,3.679208,3.693022,3.707165,3.720400,3.691407,...,3.696514,3.697459,3.682552,3.695861,3.716032,2.714534,2.349728,2.600149,3.177516,2.815382
2022-10-12 16:00:00,3.638198,3.638117,3.636924,3.651791,3.664750,3.679283,3.693103,3.707288,3.720504,3.691525,...,3.696700,3.697570,3.682751,3.696110,3.716275,2.714619,2.349809,2.600261,3.177671,2.815590


In [ ]:
from scipy.ndimage import binary_erosion,binary_dilation
def lasing_und(Ks,und_cells,rho=3e-3,rhoscale=4,verbose=False):
    kdist=rho*rhoscale
    kidxs=np.argsort(Ks)
    ksort=Ks[kidxs]
    mask=(np.abs(np.diff(ksort,prepend=-1))/ksort<kdist)|(np.abs(np.diff(ksort,append=-1))/ksort<kdist) #alls undulators who have another undulator near them in K
    mask=mask&(ksort>0.75) # Get rid of extracted undulators
    fund_idxs=kidxs[mask] #fund is for fundamental, to be distinguished from harmonics (considered later)
    
    # We should throw away any "group" of Ks with less than 5 undulators in it.
    kmin_idx=np.where(~(np.abs(np.diff(ksort,prepend=-1))/ksort<kdist)&(np.abs(np.diff(ksort,append=-1))/ksort<kdist) )[0] #idx of the lowest K in "group" of close Ks.
    #print(ksort[kmin_idx])
    for  kmin,kmax in zip(kmin_idx, list(kmin_idx[1:])+[len(kidxs)]):
        if np.sum(mask[kmin:kmax])<6:
            mask[kmin:kmax]=0
    #print(ksort[mask])
    #print(und_cells[kidxs][mask])
    fund_idxs=kidxs[mask] #fund is for fundamental, to be distinguished from harmonics (considered later)
    
    lasing_cells=und_cells[fund_idxs]
    lasing_ks=Ks[fund_idxs]
    idxs=np.argsort(lasing_cells)
    lasing_cells=lasing_cells[idxs]
    lasing_ks=lasing_ks[idxs]
    
    
    ## Check for xleap
    #Look for 4 consecutive reverse tapered undulators, with reverse taper rate larger than rho. Keep everything within that band.
    if len(lasing_cells)>4:
        if verbose: print(lasing_cells)
        mask= (np.diff(lasing_ks,append=lasing_ks[-1])/lasing_ks>(rho/2)) & (np.diff(lasing_ks,append=lasing_ks[-1])/lasing_ks<(1.5*kdist))
        if verbose: print(np.diff(lasing_ks,append=lasing_ks[-1])/lasing_ks)
        if verbose: print(mask)
        mask=binary_dilation(binary_erosion(mask,structure=[1, 1, 1, 1]),structure=[1, 1, 1, 1])
        if verbose: print(mask)
    else:
        mask=[0]
        lasing_cells=und_cells[0]
        lasing_ks=Ks[0]
    if sum(mask)>0:
        if (np.max(lasing_cells[mask])-np.min(lasing_cells[mask]))>6:
            mask=mask*0
    if sum(mask)>0:
        xleap=True
        taper_rate,taper_rate_std=(np.median(np.diff(lasing_ks[mask]))/np.mean(lasing_ks[mask]),np.std(np.diff(lasing_ks[mask]))/np.mean(lasing_ks[mask]))
        taper_ks=lasing_ks[mask]
        taper_cells=lasing_cells[mask]
        kmin=lasing_ks[mask][0]-kdist*lasing_ks[mask][0]
        kmax=lasing_ks[mask][-1]+kdist*lasing_ks[mask][-1]
        mask=(lasing_ks>kmin)&(lasing_ks<kmax)
        #diffmask=(np.abs(np.diff(lasing_ks,prepend=-1))/lasing_ks<kdist)|(np.abs(np.diff(lasing_ks,append=-1))/lasing_ks<kdist)
        #mask=mask*diffmask
    else:
        xleap=False
        taper_rate,taper_rate_std = 0,0
        mask=np.ones(np.shape(lasing_ks))==1
        taper_ks=[]
        taper_cells=[]

    data={'xleap':xleap,'taper_rate':taper_rate,'taper_rate_std':taper_rate_std,'lasing_cells':lasing_cells[mask],'lasing_ks':lasing_ks[mask],
          'taper_cells':taper_cells,'taper_ks':taper_ks}
    return data

unds = und_cells
lasing_und(kvals[0], unds)

fig = plt.figure(num=1,figsize=[3.375*1.61*2,3.375])
ax1=plt.subplot(1,2,1)
ax2=plt.subplot(1,2,2)
datas=[];
for ii,(time,kv) in enumerate(zip(times,kvals)):
    data=lasing_und(kv,unds)
    lasing_cells=data['lasing_cells']
    lasing_ks=data['lasing_ks']
    if data['xleap']:
        datas.append(data)
        datas[-1]['time']=time
        datas[-1]['idx']=ii
        #ax1.plot(lasing_cells-lasing_cells[0],(lasing_ks-lasing_ks[0])/np.median(lasing_ks))
        ax1.plot(data['taper_cells'],(data['taper_ks']-data['taper_ks'][0])/np.median(data['taper_ks']))
    else:
        ax2.plot(lasing_cells-lasing_cells[0],(lasing_ks-lasing_ks[0])/np.median(lasing_ks))

TypeError: 'NoneType' object is not iterable